In [ ]:

import math
import torch

KAPPA = 2.0  #speed that var goes back to long term level
THETA = 0.04 # long term variance
XI = 0.3 #how much variance changes around
RHO = -0.7 #stock and vol move opposite
BURN_IN = 30 #how many days of price history are simulated before option starts

def heston(n_paths, n_steps, seed, S0=100.0, T=30/252, burn_in=BURN_IN, kappa=KAPPA, theta=THETA, xi=XI, rho=RHO, substeps=10):
    dt=T/n_steps/substeps
    total_days = n_steps + burn_in
    g=torch.Generator().manual_seed(seed)
    shape, scale = 2 * kappa * theta / (xi ** 2), xi ** 2 / (2 * kappa)
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        v=torch.distributions.Gamma(shape, 1/scale).sample((n_paths,))
    log_s=torch.zeros(n_paths)
    log_prices, variances = [log_s.clone()], [v.clone()]
    for day in range(total_days):
        for _ in range(substeps):
            z1=torch.randn(n_paths, generator=g)
            z2=rho * z1 + math.sqrt(1-rho**2) * torch.randn(n_paths, generator=g)
            v_pos=v.clamp(min=0) #make sure variance is non-negative(negative values replaced with 0)
            log_s=log_s-0.5*v_pos*dt+torch.sqrt(v_pos*dt)*z1
            v=v+kappa*(theta-v)*dt+xi*torch.sqrt(v_pos*dt)*z2
        log_prices.append(log_s.clone())
        variances.append(v.clamp(min=0))
    log_prices=torch.stack(log_prices, dim=1)
    prices = S0*torch.exp(log_prices - log_prices[:,burn_in:burn_in+1])
    return prices, torch.stack(variances, dim=1)


#yearly std of the last window days of log returns for every headhing day
def realized_volatility(prices, burn_in=BURN_IN, n_steps=30, window=10):
    log_returns = torch.log(prices[:, 1:] / prices[:, :-1])
    windows= log_returns.unfold(1, window, 1)
    start = burn_in - window
    return windows[:, start:start + n_steps].std(dim=2) * math.sqrt(252)


#premium for every path from the volatility level it starts in
def heston_premium(v0, S0=100.0, K=100.0, T=30/252, kappa=KAPPA, theta=THETA):
    avg_var = theta + (v0 - theta) * (1 - math.exp(-kappa * T)) / (kappa * T)
    s= avg_var.sqrt()
    N=torch.distributions.Normal(0.0, 1.0)
    d1 = (math.log(S0 / K) +0.5 * s**2 * T) / (s*math.sqrt(T))
    d2 = d1 - s * math.sqrt(T)
    return S0 * N.cdf(d1) - K * math.exp(-r * T) * N.cdf(d2)
